# Proyecto 1: LSA con SVD
**IMT2230 — Álgebra Lineal Avanzada y Modelamiento**  
Pontificia Universidad Católica de Chile

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
import os
import re
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.dpi'] = 120

# Detectar la raiz del proyecto (carpeta que contiene corpus_wikipedia.csv).
# Esto permite ejecutar el notebook desde cualquier directorio sin romper rutas.
def find_project_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / 'corpus_wikipedia.csv').exists():
            return d
        if (d / 'Entregables').exists() and (d / 'Enunciado.md').exists():
            return d
    return here

ROOT = find_project_root()
CACHE_FILE = ROOT / 'corpus_wikipedia.csv'
FIG_DIR = ROOT / 'Entregables' / 'Figuras'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raiz del proyecto: {ROOT}")
print(f"Corpus           : {CACHE_FILE}  (existe: {CACHE_FILE.exists()})")
print(f"Figuras se guardan en: {FIG_DIR}")

---
## P1 — Elección y descripción del corpus

Usamos artículos de **Wikipedia en español** como corpus. Wikipedia es una fuente abierta, pública y completamente reproducible. Cada documento corresponde al **resumen introductorio** (*lead section*) de un artículo de Wikipedia, que equivale a un abstract enciclopédico del tema.

Seleccionamos 5 categorías temáticas bien diferenciadas y para cada una intentamos descargar hasta 60 artículos. Algunos títulos no se descargan (no existen exactamente con el nombre buscado, o no superan el umbral mínimo de longitud), de modo que el total final es **294 documentos** distribuidos como sigue:

| Categoría | Descripción | Nº documentos |
|-----------|-------------|---------------|
| `deporte`  | Deportes, competiciones, atletas | 60 |
| `ciencia`  | Física, biología, astronomía, matemáticas | 60 |
| `economia` | Macroeconomía, finanzas, comercio | 60 |
| `cultura`  | Cine, arte, literatura, música | 59 |
| `historia` | Eventos y personajes históricos | 55 |
| **Total** | | **294** |

**Fuente:** Wikipedia en español — [https://es.wikipedia.org](https://es.wikipedia.org)  
**Acceso:** API pública vía la librería `wikipedia-api` (Python).  
**Reproducibilidad:** el código descarga los artículos por título la primera vez y los cachea en `corpus_wikipedia.csv`. En ejecuciones posteriores se carga directamente del cache, garantizando resultados idénticos sin depender de la disponibilidad del servicio web.

Elegimos este corpus porque las cinco áreas tienen vocabularios marcadamente distintos (un artículo de física no comparte casi terminología con uno de fútbol), lo que constituye un buen banco de pruebas para verificar si la SVD efectivamente "descubre" la separación temática sin etiquetas.

In [ ]:
# Títulos de artículos por categoría (65-70 por categoría para asegurar 60 exitosos)
CORPUS_TITLES = {
    "deporte": [
        "Real Madrid CF", "FC Barcelona", "Atletismo", "Rafael Nadal", "Carlos Alcaraz",
        "Fernando Alonso", "Fórmula Uno", "Tour de Francia", "Copa del Mundo de Fútbol",
        "Selección española de fútbol", "Liga de Campeones de la UEFA", "Baloncesto",
        "NBA", "Liga ACB", "Ciclismo", "Natación", "Juegos Olímpicos",
        "Comité Olímpico Internacional", "Boxeo", "Voleibol", "Balonmano",
        "Rugby", "Tenis", "Roland Garros", "Wimbledon", "Open de Australia",
        "US Open de tenis", "Maratón", "Estadio Santiago Bernabéu", "Camp Nou",
        "Premier League", "Bundesliga", "Serie A", "Copa América", "Eurocopa",
        "Pelota vasca", "Escalada deportiva", "Judo", "Taekwondo",
        "Patinaje artístico", "Triatlón", "Waterpolo", "Piragüismo", "Remo",
        "Béisbol", "Hockey sobre hielo", "Curling", "Golf", "Equitación",
        "Tiro con arco", "Vela (deporte)", "Surf", "Atletismo en los Juegos Olímpicos",
        "Decatlón", "Pentatlón moderno", "Lucha olímpica", "Halterofilia",
        "Esgrima", "Salto de esquí", "Esquí alpino", "Snowboard",
        "Fútbol sala", "Tenis de mesa", "Bádminton", "Squash (deporte)"
    ],
    "ciencia": [
        "Física", "Química", "Biología", "Matemáticas", "Astronomía",
        "Albert Einstein", "Isaac Newton", "Marie Curie", "Charles Darwin", "Stephen Hawking",
        "Mecánica cuántica", "Relatividad general", "Teoría de la evolución", "ADN", "Célula",
        "Átomo", "Energía nuclear", "Agujero negro", "Sistema solar", "Universo",
        "Big Bang", "Gravitación universal", "Electromagnetismo", "Termodinámica",
        "Inteligencia artificial", "Genoma humano", "Vacuna", "Antibiótico",
        "Telescopio espacial Hubble", "Estación Espacial Internacional",
        "Galaxia", "Vía Láctea", "Supernova", "Exoplaneta",
        "Neurociencia", "Biotecnología", "Nanotecnología", "Fotosíntesis",
        "Tabla periódica de los elementos", "Reacción química",
        "Número primo", "Teorema de Pitágoras", "Geometría euclidiana",
        "Cálculo infinitesimal", "Álgebra lineal", "Estadística", "Probabilidad",
        "Informática", "Algoritmo", "Lenguaje de programación",
        "Satélite artificial", "Luna", "Marte", "Saturno", "Júpiter",
        "Cometa", "Asteroide", "Nebulosa", "Óptica", "Acústica",
        "Geología", "Oceanografía", "Meteorología", "Ecología"
    ],
    "economia": [
        "Economía", "Capitalismo", "Socialismo", "Globalización",
        "Banco Mundial", "Fondo Monetario Internacional", "Producto interior bruto",
        "Inflación", "Desempleo", "Bolsa de valores",
        "Inversión", "Deuda pública", "Impuesto", "Banco central",
        "Reserva Federal", "Banco Central Europeo", "Euro", "Dólar estadounidense",
        "Comercio internacional", "Exportación", "Importación",
        "Tratado de libre comercio", "Unión Europea",
        "Desarrollo económico", "Pobreza", "Desigualdad económica",
        "Renta básica universal", "Keynesianismo", "Monetarismo",
        "Adam Smith", "John Maynard Keynes", "Milton Friedman",
        "Crisis financiera de 2007-2008", "Gran Depresión",
        "Criptomoneda", "Bitcóin", "Cadena de bloques",
        "Empresa multinacional", "Emprendimiento", "Logística",
        "Sector primario de la economía", "Industria",
        "Turismo", "Petróleo", "Energía renovable",
        "Salario mínimo", "Sindicato", "Monopolio",
        "Demanda (economía)", "Oferta y demanda", "Mercado (economía)",
        "Microeconomía", "Macroeconomía", "Econometría",
        "Sistema financiero", "Banco comercial", "Tipo de interés",
        "Política fiscal", "Política monetaria", "Déficit público",
        "Austeridad", "Proteccionismo", "Libre comercio"
    ],
    "cultura": [
        "Cine", "Historia del cine", "Festival de Cannes",
        "Premios Óscar", "Pedro Almodóvar", "Luis Buñuel",
        "Ingmar Bergman", "Federico Fellini", "Alfred Hitchcock",
        "Stanley Kubrick", "Steven Spielberg", "Martin Scorsese",
        "Pablo Picasso", "Salvador Dalí", "Joan Miró",
        "Francisco de Goya", "Diego Velázquez", "Museo del Prado",
        "Arte contemporáneo", "Pintura", "Escultura", "Arquitectura",
        "Antoni Gaudí", "Sagrada Família", "Flamenco",
        "Ópera", "Ballet", "Jazz", "Rock", "Música clásica",
        "Ludwig van Beethoven", "Wolfgang Amadeus Mozart",
        "Literatura", "Miguel de Cervantes", "Gabriel García Márquez",
        "Pablo Neruda", "Jorge Luis Borges", "Mario Vargas Llosa",
        "Teatro", "William Shakespeare", "Federico García Lorca",
        "Premio Nobel de Literatura", "Realismo mágico",
        "Novela", "Poesía", "Fotografía", "Diseño gráfico",
        "Gastronomía española", "Patrimonio de la Humanidad",
        "UNESCO", "Museo Guggenheim Bilbao",
        "Renacimiento", "Barroco", "Impresionismo", "Cubismo",
        "Johann Sebastian Bach", "Franz Schubert", "Frédéric Chopin",
        "Ópera italiana", "Tragedia griega", "Comedia (género)"
    ],
    "historia": [
        "Segunda Guerra Mundial", "Primera Guerra Mundial",
        "Revolución francesa", "Revolución industrial",
        "Conquista de América", "Imperio romano",
        "Edad Media", "Renacimiento", "Siglo de Oro español",
        "Inquisición española", "Independencia de los Estados Unidos",
        "Revolución rusa", "Guerra Civil española", "Franquismo",
        "Transición española", "Cristóbal Colón", "Napoleón Bonaparte",
        "Simón Bolívar", "Abraham Lincoln", "Adolf Hitler",
        "Winston Churchill", "Joseph Stalin", "Mahatma Gandhi",
        "Nelson Mandela", "Guerra Fría", "Caída del muro de Berlín",
        "Holocausto", "Derechos civiles en Estados Unidos",
        "Sufragio femenino", "Colonialismo", "Esclavitud",
        "Civilización azteca", "Civilización maya", "Imperio inca",
        "Antigua Grecia", "Alejandro Magno", "Julio César",
        "Imperio otomano", "Cruzadas", "Reforma protestante",
        "Ilustración", "Imperialismo",
        "Descolonización", "Guerra de Vietnam", "Guerra de Corea",
        "Apartheid", "Naciones Unidas", "OTAN",
        "Revolución americana", "Independencia de México",
        "Revolución mexicana", "Revolución cubana",
        "Imperio español", "Felipe II de España",
        "Carlos I de España", "Isabel I de Castilla"
    ]
}

In [ ]:
DOCS_PER_CAT = 60

def descargar_corpus(titles_dict, cache_file, docs_per_cat=60):
    """Descarga articulos de Wikipedia o los carga desde cache."""
    cache_file = Path(cache_file)
    if cache_file.exists():
        print(f"Cargando corpus desde cache: {cache_file}")
        return pd.read_csv(cache_file)

    # Import perezoso: solo se necesita la primera vez.
    import wikipediaapi
    wiki = wikipediaapi.Wikipedia(
        language='es',
        user_agent='IMT2230-LSA/1.0 (proyecto academico PUC Chile)'
    )

    records = []
    for cat, titles in titles_dict.items():
        print(f"\nDescargando '{cat}'...")
        count = 0
        for title in titles:
            if count >= docs_per_cat:
                break
            try:
                page = wiki.page(title)
                if page.exists() and len(page.summary.strip()) > 150:
                    records.append({
                        'title': page.title,
                        'text': page.summary,
                        'topic': cat
                    })
                    count += 1
                    if count % 15 == 0:
                        print(f"  {count}/{docs_per_cat}")
                time.sleep(0.08)
            except Exception as e:
                print(f"  Error en '{title}': {e}")
        print(f"  Completado: {count} articulos")

    df = pd.DataFrame(records)
    df.to_csv(cache_file, index=False)
    print(f"\nCorpus guardado en '{cache_file}'")
    return df

df = descargar_corpus(CORPUS_TITLES, CACHE_FILE, DOCS_PER_CAT)
print(f"\nCorpus final: {len(df)} documentos")
print(df['topic'].value_counts())

In [ ]:
# Vista previa del corpus
df[['title', 'topic']].groupby('topic').head(3).reset_index(drop=True)

In [ ]:
# Longitud promedio de texto por categoría
df['n_chars'] = df['text'].str.len()
df['n_words'] = df['text'].str.split().str.len()
print(df.groupby('topic')[['n_chars', 'n_words']].mean().round(0).to_string())

---
## P2 — Hipótesis inicial

El corpus está compuesto por artículos de cinco categorías temáticas claramente delimitadas: deporte, ciencia, economía, cultura e historia. **Nuestra hipótesis es que la SVD capturará estas diferencias temáticas en sus primeras componentes**, dado que el vocabulario de cada categoría es bastante distinto (por ejemplo, *gol* y *partido* son específicos de deportes; *átomo* y *energía* de ciencia; *inflación* y *PIB* de economía).

En particular, esperamos que:

1. Los documentos de la misma categoría queden **agrupados en clusters** en el espacio de baja dimensión.
2. Los **términos característicos** de cada área aparezcan en regiones separadas del mapa de términos en 2D.
3. Los **primeros 5–10 valores singulares** concentren una fracción importante de la energía total, evidenciando que existe estructura temática dominante.
4. La categoría **historia** podría solaparse parcialmente con **política/economía**, ya que comparten vocabulario histórico-político.

---
## P3 — Preprocesamiento del texto

Aplicamos el siguiente pipeline de limpieza:

1. **Minúsculas**: normalizamos toda la cadena a *lowercase*.
2. **Eliminación de caracteres no alfabéticos**: conservamos letras con tilde y la ñ, eliminamos números, signos de puntuación y dígitos.
3. **Eliminación de stopwords**: usamos una **lista manual de stopwords en español** (artículos, preposiciones, conjunciones, pronombres y formas verbales auxiliares frecuentes), construida a partir de listas estándar y *ampliada con términos de alta frecuencia propios de la redacción enciclopédica* que aportan poco significado temático (*denominado*, *conocido*, *llamado*, *también*, *así*, *encuentra*, *trata*, *refiere*, etc.). Estos términos aparecen masivamente en Wikipedia y, sin filtrarlos, dominan las primeras componentes de la SVD sin aportar contenido temático.
4. **Filtro por longitud**: descartamos tokens de menos de 3 caracteres (mayoritariamente ruido residual: *de*, *al*, *el*, etc., ya eliminados por la lista de stopwords pero hay casos como `'el'` sin tilde que se cuelan).
5. **Normalización de tildes para el filtrado** (la comparación contra stopwords se hace sin tildes para no depender del *encoding* exacto del texto fuente).

Después de la limpieza pasamos a TF-IDF con `TfidfVectorizer` de scikit-learn, configurado así:

- `sublinear_tf=True` → reemplaza $\text{tf}$ por $1 + \log(\text{tf})$, comprimiendo frecuencias altas (un término que aparece 50 veces no pesa 50× más que uno que aparece 1 vez).
- `min_df=3` → ignora términos que aparecen en menos de 3 documentos (descarta hápax y errores de tipeo).
- `max_df=0.80` → ignora términos presentes en más del 80% de documentos (acción complementaria al filtro de stopwords).
- `max_features=3000` → conserva las 3 000 palabras más informativas según TF-IDF global.
- `norm='l2'` → cada vector de documento se normaliza a norma euclídea 1.

La entrada $A_{ij}$ del peso TF-IDF mide cuán relevante es el término $i$ para el documento $j$: favorece términos frecuentes en ese documento pero raros en el corpus, que es exactamente el comportamiento que necesitamos para que la SVD encuentre estructura temática.

In [ ]:
# Lista manual de stopwords en espanol (no requiere descargas externas).
# Base: articulos, preposiciones, conjunciones, pronombres y verbos auxiliares
# de uso muy frecuente. La comparacion se hace sobre tokens sin tilde.
STOPWORDS_BASE = {
    'a', 'al', 'algo', 'algun', 'alguna', 'algunas', 'alguno', 'algunos',
    'ante', 'antes', 'aqui', 'aquel', 'aquella', 'aquellas', 'aquello',
    'aquellos', 'asi', 'aun', 'aunque', 'bajo', 'bien', 'cada', 'casi',
    'como', 'con', 'contra', 'cual', 'cuales', 'cualquier', 'cuando',
    'cuanto', 'cuya', 'cuyas', 'cuyo', 'cuyos', 'de', 'del', 'demas',
    'desde', 'donde', 'dos', 'durante', 'el', 'ella', 'ellas', 'ello',
    'ellos', 'en', 'entre', 'era', 'eran', 'eras', 'eres', 'es', 'esa',
    'esas', 'ese', 'eso', 'esos', 'esta', 'estaba', 'estaban', 'estado',
    'estamos', 'estan', 'estas', 'este', 'esto', 'estos', 'estoy',
    'fue', 'fuera', 'fueron', 'ha', 'haber', 'habia', 'habian', 'han',
    'has', 'hasta', 'hay', 'haya', 'he', 'hemos', 'hizo', 'la', 'las',
    'le', 'les', 'lo', 'los', 'mas', 'me', 'mediante', 'mi', 'mis',
    'mismo', 'misma', 'mismos', 'mismas', 'mucho', 'muchos', 'muy',
    'nada', 'ni', 'no', 'nos', 'nosotros', 'nuestra', 'nuestras',
    'nuestro', 'nuestros', 'nunca', 'o', 'os', 'otra', 'otras', 'otro',
    'otros', 'para', 'pero', 'poco', 'por', 'porque', 'puede', 'pueden',
    'que', 'quien', 'quienes', 'se', 'sea', 'sean', 'segun', 'ser',
    'sera', 'seran', 'si', 'sido', 'siendo', 'sin', 'sobre', 'solo',
    'son', 'soy', 'su', 'sus', 'tal', 'tambien', 'tan', 'tanto', 'te',
    'tener', 'tengo', 'ti', 'tiene', 'tienen', 'toda', 'todas', 'todo',
    'todos', 'tras', 'tu', 'tus', 'un', 'una', 'unas', 'uno', 'unos',
    'vez', 'y', 'ya', 'yo'
}

# Stopwords adicionales especificas de la redaccion enciclopedica de Wikipedia.
# Estas palabras aparecen mucho en los resumenes pero NO indican un tema
# (denominado, conocido, etc.). Si no las filtramos, dominan la primera
# componente de la SVD sin aportar significado.
STOPWORDS_ENCICLOPEDICAS = {
    'denominado', 'denominada', 'denominados', 'denominadas',
    'conocido', 'conocida', 'conocidos', 'conocidas',
    'llamado', 'llamada', 'llamados', 'llamadas',
    'dicho', 'dicha', 'dichos', 'dichas',
    'determinado', 'determinada', 'determinados', 'determinadas',
    'propio', 'propia', 'propios', 'propias',
    'primer', 'primera', 'segundo', 'segunda', 'tercer', 'tercera',
    'nuevo', 'nueva', 'distinto', 'distinta', 'distintos', 'distintas',
    'diferente', 'diferentes', 'general', 'gran', 'grandes',
    'parte', 'partes', 'forma', 'manera', 'tipo', 'tipos',
    'trata', 'refiere', 'encuentra', 'considera', 'utiliza', 'usa',
    'emplea', 'obtiene', 'realiza', 'hace', 'puede', 'pueden',
    'teniendo', 'tenido', 'haya', 'siendo', 'sido',
    'respecto', 'mientras', 'embargo', 'ademas',
    'hecho', 'caso', 'ejemplo', 'numero', 'nombre',
    'ver', 'dado'
}

STOPWORDS_ES = STOPWORDS_BASE | STOPWORDS_ENCICLOPEDICAS
print(f"Stopwords base       : {len(STOPWORDS_BASE)}")
print(f"Stopwords enciclop.  : {len(STOPWORDS_ENCICLOPEDICAS)}")
print(f"Total stopwords      : {len(STOPWORDS_ES)}")


def quitar_tildes(s):
    for a, b in [('á','a'),('é','e'),('í','i'),('ó','o'),('ú','u'),('ü','u'),('ñ','n')]:
        s = s.replace(a, b)
    return s


def preprocess(text):
    text = text.lower()
    # Conservar letras (incluyendo tildes y ñ), descartar el resto
    text = re.sub(r'[^a-zà-ÿñ\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = []
    for orig in text.split():
        norm = quitar_tildes(orig)
        if len(norm) > 2 and norm not in STOPWORDS_ES:
            tokens.append(orig)
    return ' '.join(tokens)


df['text_clean'] = df['text'].apply(preprocess)

print("\nPreprocesamiento completado.")
print(f"\nORIGINAL: {df['text'].iloc[0][:280]}")
print(f"\nLIMPIO  : {df['text_clean'].iloc[0][:280]}")

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    min_df=3,
    max_df=0.80,
    sublinear_tf=True,
    norm='l2'
)

X_sklearn = vectorizer.fit_transform(df['text_clean'])   # (n_docs × n_terms)
vocabulary = vectorizer.get_feature_names_out()

print(f"Dimensiones sklearn (docs × términos): {X_sklearn.shape}")
print(f"Tamaño del vocabulario final         : {len(vocabulary)} términos")
print(f"Densidad de la matriz                : {X_sklearn.nnz / np.prod(X_sklearn.shape) * 100:.2f}%")
print(f"\nMuestra del vocabulario: {list(vocabulary[:30])}")

---
## P4 — Construcción de la matriz textual

Seguimos la convención del curso: construimos una matriz **término-documento**

$$A \in \mathbb{R}^{m \times n}$$

donde las **filas** representan términos ($m$ términos) y las **columnas** representan documentos ($n$ documentos). scikit-learn entrega la transpuesta (documentos × términos), por lo que transponemos.

La entrada $A_{ij}$ es el peso **TF-IDF** del término $i$ en el documento $j$:

$$A_{ij} = \bigl(1 + \log \text{tf}_{ij}\bigr) \cdot \log\frac{N+1}{\text{df}_i + 1}$$

con normalización L2 por columna (documento), de modo que cada columna tiene norma 1.

In [ ]:
# Transponer para seguir convención término-documento del curso
A = X_sklearn.T.toarray()   # (m × n)

m, n = A.shape
print(f"Matriz A (término-documento):")
print(f"  Shape: {A.shape}")
print(f"  m = {m} términos   (filas)")
print(f"  n = {n} documentos (columnas)")
print(f"  Valores no nulos: {np.count_nonzero(A):,} de {m*n:,} ({np.count_nonzero(A)/m/n*100:.2f}%)")
print(f"\nA_ij = TF-IDF del término i en el documento j")

---
## P5 — Cálculo de la SVD

Calculamos la descomposición en valores singulares de $A$:

$$A = U \Sigma V^T$$

- $U \in \mathbb{R}^{m \times r}$: columnas ortogonales, cada columna $U_{:,k}$ es una **dirección principal en el espacio de términos** ("concepto latente").
- $\Sigma \in \mathbb{R}^{r \times r}$: diagonal con $\sigma_1 \geq \sigma_2 \geq \ldots \geq \sigma_r \geq 0$, donde $\sigma_k$ mide la **importancia del concepto $k$**.
- $V^T \in \mathbb{R}^{r \times n}$: la fila $k$ indica cómo se proyectan los **documentos** sobre el concepto $k$.

Usamos `full_matrices=False` para obtener la **SVD reducida**: $r = \min(m,n) = n$, lo que evita almacenar la $U$ completa de $m \times m$.

La aproximación de rango $k$ es:
$$A_k = U_k \Sigma_k V_k^T \approx A, \quad \|A - A_k\|_F = \sqrt{\sigma_{k+1}^2 + \ldots + \sigma_r^2}$$

In [ ]:
print("Calculando SVD de A (full_matrices=False)...")
U, s, Vt = np.linalg.svd(A, full_matrices=False)

print(f"\nA = U Σ Vᵀ (SVD reducida):")
print(f"  U  : {U.shape}    → espacio de términos")
print(f"  s  : {s.shape}     → valores singulares")
print(f"  Vt : {Vt.shape}   → espacio de documentos")
print(f"\nPrimeros 15 valores singulares:")
print(np.round(s[:15], 4))

# Verificación numérica
A_rec = U @ np.diag(s) @ Vt
print(f"\nError de reconstrucción ||A - UΣVᵀ||_F = {np.linalg.norm(A - A_rec):.2e}  (debe ser ~0)")

In [ ]:
print("=== Interpretación de U, Σ, V en LSA ===")
print()
print("σₖ (k-ésimo valor singular):")
print("  Importancia del concepto latente k.")
print("  σ₁ >> σ₂ indica un tema muy dominante en el corpus.")
print()
print("U[:,k] (k-ésimo vector singular izquierdo):")
print("  Dirección en el espacio de términos del concepto k.")
print("  Términos con mayor |U[i,k]| son los más representativos del concepto k.")
print()
print("Vᵀ[k,:] (k-ésima fila de Vᵀ):")
print("  Proyección de los documentos sobre el concepto k.")
print("  Documentos con mayor |Vᵀ[k,j]| son los más representativos del concepto k.")

---
## P6 — Valores singulares y elección de dimensión

Graficamos el decaimiento de los valores singulares y la energía acumulada para elegir $k$:

$$\text{Energía acumulada}(k) = \frac{\sum_{i=1}^{k} \sigma_i^2}{\sum_{i=1}^{n} \sigma_i^2}$$

In [ ]:
energy = np.cumsum(s**2) / np.sum(s**2)
k = 20

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Valores singulares
axes[0].plot(range(1, 51), s[:50], 'o-', markersize=4, color='steelblue')
axes[0].axvline(x=k, color='crimson', linestyle='--', alpha=0.8, label=f'k = {k}')
axes[0].set_xlabel('Índice $k$', fontsize=12)
axes[0].set_ylabel('Valor singular $\\sigma_k$', fontsize=12)
axes[0].set_title('Decaimiento de valores singulares', fontsize=13)
axes[0].grid(alpha=0.3)
axes[0].legend()

# Energia acumulada
axes[1].plot(range(1, len(s)+1), energy * 100, color='darkorange')
axes[1].axhline(y=80, color='gray', linestyle=':', label='80 %')
axes[1].axvline(x=k, color='crimson', linestyle='--', alpha=0.8, label=f'k = {k}')
axes[1].set_xlabel('Número de componentes $k$', fontsize=12)
axes[1].set_ylabel('Energía acumulada (%)', fontsize=12)
axes[1].set_title('Energía acumulada de la SVD', fontsize=13)
axes[1].set_xlim(0, min(200, len(s)))
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'valores_singulares.png', bbox_inches='tight')
plt.show()

# Resumen numerico
k_50  = int(np.searchsorted(energy, 0.50)) + 1
k_80  = int(np.searchsorted(energy, 0.80)) + 1
print(f"sigma_1 = {s[0]:.3f}   sigma_2 = {s[1]:.3f}   sigma_5 = {s[4]:.3f}   sigma_20 = {s[19]:.3f}")
print(f"Energia acumulada con k = {k:>3}: {energy[k-1]*100:5.1f} %")
print(f"Energia acumulada con k =   5: {energy[4]*100:5.1f} %")
print(f"Para alcanzar 50% se necesitan k = {k_50} componentes")
print(f"Para alcanzar 80% se necesitan k = {k_80} componentes")

**Lectura del decaimiento y elección de $k$.** El gráfico muestra dos rasgos importantes:

1. **Caída brusca inicial**: $\sigma_1 \approx 3.56$ pasa a $\sigma_2 \approx 2.23$ (caída del 37%). Las primeras 5–6 componentes marcan un cambio claro en el ritmo del decaimiento.
2. **Decaimiento lento posterior**: a partir de $\sigma_7$ en adelante la curva es casi lineal. Esto es típico de corpus con muchos temas y vocabulario rico: no existe una sola dirección que concentre toda la varianza; la información se reparte entre muchos modos.

Como consecuencia, la **energía acumulada** crece despacio: con $k = 20$ capturamos ≈ 21% de la varianza total y para alcanzar el 80% se necesitan ≈ 183 componentes (más de la mitad del rango). Esto **invalida** parcialmente el punto 3 de nuestra hipótesis inicial (esperábamos que las primeras 5–10 componentes concentraran una fracción importante de la energía).

**¿Por qué entonces $k = 20$ es razonable?** Porque para los usos típicos de LSA — clustering, búsqueda de vecinos cercanos, visualización — *no* necesitamos reconstruir la matriz con baja error; necesitamos un espacio donde documentos del mismo tema queden cerca y documentos distintos queden lejos. Con $k = 20$:

- Las componentes 2–5 ya capturan las direcciones temáticas principales (lo verificaremos en P8).
- El espacio es lo bastante reducido para ser interpretable y para que el cálculo de similitud coseno entre documentos sea barato.
- La calidad de las vecindades semánticas en este espacio es excelente (sim cos > 0.9 entre documentos del mismo tema), pese a la energía modesta.

En síntesis: la *energía capturada* y la *utilidad para clustering semántico* son cosas distintas; el corpus es de rango efectivo alto, pero la **estructura temática** vive en muy pocas direcciones.

---
## P7 — Representación de documentos y términos en baja dimensión

Para visualizar en 2D usamos las dos primeras componentes:

$$A_2 = U_2 \Sigma_2 V_2^T$$

- **Coordenadas de documentos**: $D_2 = \Sigma_2 V_2^T \in \mathbb{R}^{2 \times n}$
- **Coordenadas de términos**: $T_2 = U_2 \Sigma_2 \in \mathbb{R}^{m \times 2}$

Ambas representaciones viven en el mismo espacio semántico de 2 dimensiones: documentos y términos pueden compararse directamente.

In [ ]:
U2  = U[:, :2]
s2  = s[:2]
Vt2 = Vt[:2, :]

D2 = np.diag(s2) @ Vt2     # (2, n)
T2 = U2 @ np.diag(s2)      # (m, 2)

print(f"D2 (documentos en 2D): {D2.shape}")
print(f"T2 (términos en 2D)  : {T2.shape}")

In [ ]:
# --- Visualizacion 2D de documentos (componentes 1 y 2) ---
unique_cats = sorted(df['topic'].unique())
palette = sns.color_palette("tab10", len(unique_cats))
cat_to_color = {cat: palette[i] for i, cat in enumerate(unique_cats)}

fig, ax = plt.subplots(figsize=(11, 8))
for cat in unique_cats:
    idx = df.index[df['topic'] == cat]
    ax.scatter(D2[0, idx], D2[1, idx],
               color=cat_to_color[cat], alpha=0.70, s=50, label=cat,
               edgecolors='white', linewidths=0.5)

ax.axhline(0, color='k', linewidth=0.4, alpha=0.3)
ax.axvline(0, color='k', linewidth=0.4, alpha=0.3)
ax.legend(title='Categoría', fontsize=10, loc='best')
ax.set_xlabel(f'Componente 1  ($\\sigma_1 = {s[0]:.2f}$)', fontsize=12)
ax.set_ylabel(f'Componente 2  ($\\sigma_2 = {s[1]:.2f}$)', fontsize=12)
ax.set_title('Representación 2D de documentos\n(componentes 1 y 2 de la SVD)', fontsize=13)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIG_DIR / 'documentos_2d.png', bbox_inches='tight')
plt.show()

**Lectura del gráfico de documentos.** Tres observaciones clave:

1. **La componente 1 no separa por tema**: todos los documentos tienen coordenadas negativas en este eje, lo que indica que esta dirección captura sobre todo una *magnitud común* de los vectores TF-IDF (lo confirmaremos en P8 con los términos asociados al polo (+), que son palabras raras y heterogéneas).
2. **La componente 2 sí es discriminativa**: en la zona superior se agrupan los documentos de **ciencia** (azul) y, mezclados con ellos, parte de **economía** (rojo); en la zona inferior se concentran **historia** (morado, claramente separada hacia la izquierda inferior) y **deporte** (verde, en la franja central inferior).
3. **Cultura** (naranja) es la más dispersa: ocupa la zona central porque internamente mezcla cine, pintura, música clásica, literatura y arquitectura — subtemas que comparten poco vocabulario entre sí. **Historia** es la categoría con clústeres más cohesivos (esto se cuantificará en el heatmap).

Es decir, ya con 2 dimensiones se distingue visualmente la estructura temática del corpus, aunque las direcciones útiles son la 2ª y siguientes, no la primera.


In [ ]:
# --- Visualizacion 2D de terminos relevantes ---
# Estrategia de seleccion:
#   - Top-N terminos por mayor C2 positiva (polo "ciencia")
#   - Top-N terminos por mayor C2 negativa (polo "historia/politica")
#   - Top-N terminos por mayor |C1| (los terminos que mas se alejan
#     del origen en la direccion generica). Esto evita una nube
#     ilegible cerca de C1=0 (donde se aglomeran terminos raros del polo + de C1).
N = 15
top_pos_c2 = np.argsort(T2[:, 1])[-N:]
top_neg_c2 = np.argsort(T2[:, 1])[:N]
top_neg_c1 = np.argsort(T2[:, 0])[:N]
idx_show = np.unique(np.concatenate([top_pos_c2, top_neg_c2, top_neg_c1]))

fig, ax = plt.subplots(figsize=(13, 9))
ax.scatter(T2[idx_show, 0], T2[idx_show, 1], alpha=0.4, s=24, color='slategray')
for idx in idx_show:
    ax.annotate(vocabulary[idx],
                (T2[idx, 0], T2[idx, 1]),
                fontsize=9.5, alpha=0.9,
                xytext=(5, 4), textcoords='offset points')

ax.axhline(0, color='k', linewidth=0.4, alpha=0.3)
ax.axvline(0, color='k', linewidth=0.4, alpha=0.3)
ax.set_xlabel(f'Componente 1  ($\\sigma_1 = {s[0]:.2f}$)', fontsize=12)
ax.set_ylabel(f'Componente 2  ($\\sigma_2 = {s[1]:.2f}$)', fontsize=12)
ax.set_title('Representacion 2D de terminos relevantes\n(extremos en C2 y mas negativos en C1)', fontsize=13)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIG_DIR / 'terminos_2d.png', bbox_inches='tight')
plt.show()


**Lectura del gráfico de términos.** El mapa de términos comparte la misma geometría que el de documentos (ambos viven en el espacio de $U\Sigma$ y $\Sigma V^T$), lo que permite interpretarlos en conjunto:

- **Polo superior** (Componente 2 positiva): `teoría, estudia, sistemas, materia, energía, física, propiedades, química, fenómenos, procesos` — vocabulario de **ciencia**.
- **Polo inferior** (Componente 2 negativa): `guerra, imperio, españa, francia, europa, militar, gobierno, política, república, ejército` — vocabulario de **historia / política**.
- **Lado derecho** (Componente 1 cercana a 0): términos muy específicos y de baja frecuencia global; **lado izquierdo** (componente 1 fuertemente negativa): términos comunes (`siglo, años, países, hasta, otros`).

La coincidencia entre la región donde caen los documentos de ciencia (zona superior del scatter de docs) y la región donde caen los términos científicos (zona superior del scatter de términos) confirma que ambas representaciones son consistentes: un documento de física está cerca de las palabras *teoría* y *materia* porque ese documento las usa con peso TF-IDF alto.


In [ ]:
# --- Visualizacion complementaria: documentos en componentes 2 y 3 ---
# Como se vera abajo, la componente 1 captura un eje "magnitud/longitud"
# mas que un tema. Las componentes 2 y 3 son las primeras dos direcciones
# claramente tematicas, asi que vale la pena visualizarlas tambien.
D23 = np.vstack([s[1] * Vt[1, :], s[2] * Vt[2, :]])  # (2, n)

fig, ax = plt.subplots(figsize=(11, 8))
for cat in unique_cats:
    idx = df.index[df['topic'] == cat]
    ax.scatter(D23[0, idx], D23[1, idx],
               color=cat_to_color[cat], alpha=0.75, s=50, label=cat,
               edgecolors='white', linewidths=0.5)

ax.axhline(0, color='k', linewidth=0.4, alpha=0.3)
ax.axvline(0, color='k', linewidth=0.4, alpha=0.3)
ax.legend(title='Categoría', fontsize=10, loc='best')
ax.set_xlabel(f'Componente 2  ($\\sigma_2 = {s[1]:.2f}$)', fontsize=12)
ax.set_ylabel(f'Componente 3  ($\\sigma_3 = {s[2]:.2f}$)', fontsize=12)
ax.set_title('Representación 2D de documentos\n(componentes 2 y 3, las primeras direcciones temáticas)', fontsize=13)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FIG_DIR / 'documentos_2d_c2c3.png', bbox_inches='tight')
plt.show()

**Lectura del gráfico (C2, C3).** Cuando *descartamos* la componente 1 (que captura "magnitud" y no tema) y miramos directamente los dos primeros ejes temáticos, los cinco clusters se vuelven mucho más nítidos:

- **Ciencia** (azul) se separa por la mitad superior (C2 positiva, vocabulario de teoría/materia/energía).
- **Deporte** (verde) se separa por la derecha (C3 positiva, vocabulario de juegos/equipos/competiciones).
- **Historia** (morado) y **economía** (rojo) ocupan el cuadrante inferior-izquierdo, lo que vuelve a anticipar el solapamiento histórico-económico que la matriz de similitud (figura siguiente) cuantifica.
- **Cultura** (naranja) sigue siendo la más dispersa, pero ahora se la ve concentrada principalmente en el cuadrante inferior-derecho cuando se mueve hacia C3 negativa y C2 cerca de cero.

Esta figura ilustra de manera más limpia lo que la SVD descubre: cada componente más allá de la primera codifica un *contraste* entre pares de temas (C2: ciencia vs historia; C3: deporte vs política/economía; etc.).


---
## P8 — Exploración semántica

Usamos la representación de rango $k$ para tres tipos de exploración:
1. **Términos que caracterizan cada componente** (qué concepto captura cada dirección singular).
2. **Documentos más cercanos** a un documento dado (similitud coseno en el espacio reducido).
3. **Términos más cercanos** a una palabra de consulta (vecindad semántica latente).

In [ ]:
# --- Ejemplo 1: términos que caracterizan las primeras 5 componentes ---
def top_terms_componente(U, vocabulary, s, comp_idx, n=12):
    col = U[:, comp_idx]
    pos = np.argsort(col)[-n:][::-1]
    neg = np.argsort(col)[:n]
    print(f"\n{'─'*65}")
    print(f" Componente {comp_idx+1}  (σ_{comp_idx+1} = {s[comp_idx]:.3f})")
    print(f"{'─'*65}")
    print(f"  Polo (+): {', '.join(vocabulary[i] for i in pos)}")
    print(f"  Polo (−): {', '.join(vocabulary[i] for i in neg)}")

print("Términos con mayor peso en las primeras componentes de U:")
for ci in range(5):
    top_terms_componente(U, vocabulary, s, ci)

**¿Qué tema captura cada componente?** Los polos de las primeras 5 componentes son interpretables y coherentes con las categorías del corpus:

| Comp. | Polo (+) interpretación | Polo (−) interpretación |
|:-----:|------------------------|-------------------------|
| 1 | términos raros/específicos | términos comunes y genéricos (eje no temático) |
| 2 | **ciencia** (teoría, materia, energía, física) | **historia / política** (guerra, imperio, militar) |
| 3 | **deporte** (juegos, olímpicos, equipo, fútbol) | **economía / política** (economía, gobierno, crisis) |
| 4 | **cultura** (música, arte, cine, literatura) | **economía** (servicios, bienes, mercado, monetaria) |
| 5 | **ciencia/astronomía** (universo, galaxias, planetas) | **cultura / economía** (música, cine, comercio) |

Es importante notar que **5 categorías no equivalen a 5 componentes**: la SVD encuentra ejes ortogonales que oponen pares de temas, no clases individuales. Por ejemplo, el eje 2 contrasta ciencia con historia, y el eje 4 contrasta cultura con economía. Las cinco categorías quedan así codificadas distribuidamente en las componentes 2–5.


In [ ]:
# --- Ejemplo 2: documentos más cercanos ---
Dk = np.diag(s[:k]) @ Vt[:k, :]   # (k, n)

def cosine_sim(X):
    norms = np.linalg.norm(X, axis=0, keepdims=True)
    norms[norms == 0] = 1.0
    Xn = X / norms
    return Xn.T @ Xn

sim_docs = cosine_sim(Dk)

def documentos_cercanos(idx, sim, df, top_n=5):
    sims = sim[idx].copy()
    sims[idx] = -1
    top = np.argsort(sims)[-top_n:][::-1]
    print(f"\nDocumento base [{df['topic'].iloc[idx]}]:")
    print(f"  '{df['title'].iloc[idx]}'")
    print("  Más cercanos:")
    for j in top:
        print(f"    [{df['topic'].iloc[j]:10s}] (sim={sims[j]:.3f})  {df['title'].iloc[j]}")

for cat in unique_cats:
    idx = df[df['topic'] == cat].index[0]
    documentos_cercanos(idx, sim_docs, df)

**Documentos cercanos: vecindades coherentes.** Para cada categoría tomamos el primer documento y revisamos sus 5 vecinos más cercanos en el espacio de rango $k=20$. En **todos** los casos los 5 vecinos pertenecen a la *misma* categoría (similitud coseno > 0.84 en todos los casos, > 0.95 para los primeros vecinos). Algunos ejemplos especialmente coherentes:

- **`Física`** → `Electromagnetismo` (0.92), `Mecánica cuántica` (0.91), `Termodinámica` (0.90), `Óptica` (0.90): los principales sub-campos de la física clásica/moderna.
- **`Real Madrid CF`** → `FC Barcelona` (0.98), `Premier League` (0.96), `Liga de Campeones de la UEFA` (0.91), `Selección de fútbol de España` (0.86), `Eurocopa` (0.85): todo el ecosistema institucional del fútbol europeo.
- **`Cine`** → `Historia del cine` (0.96), `Festival de Cannes` (0.80), `Alfred Hitchcock` (0.80), `Premios Óscar` (0.80), `Ingmar Bergman` (0.75).
- **`Segunda Guerra Mundial`** → `Primera Guerra Mundial`, `Guerra de Corea`, `Guerra de Vietnam`, `Guerra Fría`, `Dictadura de Francisco Franco`: todos conflictos del siglo XX.

Sin haber dado ninguna etiqueta, LSA recupera vecinos temáticamente correctos. Esta es la propiedad fundamental que hace útil a la representación reducida para tareas de recuperación de información.


In [ ]:
# --- Ejemplo 3: términos más cercanos a una palabra de consulta ---
Tk_full = U[:, :k] @ np.diag(s[:k])   # (m, k)
vocab_list = list(vocabulary)

def terminos_cercanos(query, Tk, vocab_list, top_n=8):
    if query not in vocab_list:
        print(f"'{query}' no está en el vocabulario.")
        return
    idx = vocab_list.index(query)
    vec = Tk[idx]
    norms = np.linalg.norm(Tk, axis=1)
    sims = Tk @ vec / (norms * np.linalg.norm(vec) + 1e-10)
    top = np.argsort(sims)[-top_n-1:][::-1][1:]
    print(f"\nTérminos más cercanos a '{query}':")
    for i in top:
        print(f"  {vocabulary[i]:22s}  sim={sims[i]:.3f}")

for word in ['gravedad', 'guerra', 'inflación', 'película', 'fútbol']:
    terminos_cercanos(word, Tk_full, vocab_list)

**Términos cercanos: vecindad semántica latente.** La SVD recupera relaciones semánticas que el TF-IDF puro no expresaría:

- `gravedad` → `agujeros, negros, gravitatoria, curvatura, dimensiones, cósmica` — el contexto astrofísico está bien capturado (curvatura espacio-temporal, agujeros negros).
- `guerra` → `conferencia, conflicto, militar, derrota, aliados, contienda` — campo semántico militar completo.
- `inflación` → `monetarias, tasa, recesión, crédito, fiscal, dinero` — terminología macroeconómica.
- `película` → `óscar, festival, cineasta, cannes, documental, guionista` — universo cinematográfico.
- `fútbol` → `fifa, iffhs, clubes, uefa, league, temporada` — instituciones y vocabulario futbolístico.

Notar que ninguna de estas asociaciones está hard-codeada: emergen automáticamente del hecho de que las palabras aparecen en los mismos contextos documentales. LSA captura **co-ocurrencia indirecta** (palabras que comparten *vecinos* de co-ocurrencia, aunque no co-ocurran directamente).


In [ ]:
# --- Heatmap de similitud promedio entre categorias ---
cat_sim = pd.DataFrame(index=unique_cats, columns=unique_cats, dtype=float)
for c1 in unique_cats:
    for c2 in unique_cats:
        i1 = df.index[df['topic'] == c1]
        i2 = df.index[df['topic'] == c2]
        block = sim_docs[np.ix_(i1, i2)].copy()
        if c1 == c2:
            np.fill_diagonal(block, np.nan)
        cat_sim.loc[c1, c2] = np.nanmean(block)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cat_sim.astype(float), annot=True, fmt='.3f',
            cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title(f'Similitud coseno promedio entre categorias (k={k})', fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / 'similitud_categorias.png', bbox_inches='tight')
plt.show()

print('\nResumen del heatmap:')
print(f'  Diagonal promedio (intra-categoria): {np.nanmean(np.diag(cat_sim.values)):.3f}')
off = cat_sim.values.copy()
np.fill_diagonal(off, np.nan)
print(f'  Off-diagonal promedio (inter-cat.) : {np.nanmean(off):.3f}')
pair = np.unravel_index(np.nanargmax(off), off.shape)
print(f'  Par cruzado mas alto: {cat_sim.index[pair[0]]}-{cat_sim.columns[pair[1]]} = {off[pair]:.3f}')


**Lectura del heatmap.** La diagonal (promedio 0.422) es **≈ 3× mayor** que la similitud cruzada promedio (0.145), lo que confirma cuantitativamente la separación temática del corpus:

- **historia** es la más cohesiva (diagonal 0.524): los textos sobre guerras, imperios y eventos políticos comparten un núcleo léxico muy estable (`guerra, imperio, militar, gobierno, revolución`).
- **economía** (0.452) y **ciencia** (0.394) también son cohesivas.
- **deporte** (0.381) y **cultura** (0.358) son las menos cohesivas: deporte mezcla deportes muy distintos (fútbol, tenis, ciclismo, ajedrez, esquí), cultura mezcla cine, pintura, música y literatura — subdominios con vocabularios poco solapados.

Fuera de la diagonal, el patrón confirma la **hipótesis 4** de P2: el par **historia–economía** (0.210) es el cruzado más alto, seguido de **cultura–historia** (0.192). Los pares más disjuntos son **deporte–economía** (0.100) y **deporte–ciencia** (0.100): consistente con que el vocabulario del deporte tiene poco solape con macroeconomía o ciencias naturales.


---
## P9 — Discusión, limitaciones y conclusiones

### Comparación con la hipótesis inicial

Repasamos los cuatro puntos de la hipótesis (P2) frente a los resultados observados:

**(1) Clusters por categoría en 2D — se cumple, pero no en C1.**  
La figura `documentos_2d.png` muestra que en (C1, C2) todos los documentos caen en el semiplano $x < 0$: la primera componente actúa como un eje "genérico" que captura una magnitud común de los vectores TF-IDF normalizados, no un tema. La separación temática real aparece en la **componente 2**: arriba se concentran ciencia y economía, abajo historia y deporte. Cuando re-graficamos los documentos en (C2, C3) — figura `documentos_2d_c2c3.png` — los cinco clusters se vuelven mucho más nítidos: deporte arriba a la izquierda, ciencia a la derecha, historia abajo a la izquierda, economía abajo a la derecha, y cultura como una banda intermedia central. Es decir, la hipótesis 1 se confirma plenamente cuando se mira **a partir de la componente 2**.

**(2) Términos temáticos en regiones separadas — se cumple a partir de la componente 2.**  
Las componentes 2–5 son **claramente temáticas**:

- **Componente 2** opone *ciencia* (`teoría, sistemas, materia, energía, física, química`) a *historia/política* (`guerra, imperio, francia, militar, gobierno`).
- **Componente 3** opone *deporte* (`juegos, olímpicos, jugadores, equipo, fútbol`) a *economía/política* (`economía, gobierno, económica, revolución, crisis`).
- **Componente 4** opone *cultura* (`música, arte, cine, literatura, pintura`) a *economía* (`servicios, bienes, mercado, comercio, monetaria`).
- **Componente 5** opone *ciencia/astronomía* a *cultura/economía*.

Cada componente latente codifica un eje semántico interpretable: las cinco categorías quedan codificadas *distribuidamente* en estas componentes (no hay una correspondencia 1-a-1 categoría-componente, sino contrastes entre pares).

**(3) Concentración de energía en pocas componentes — NO se cumple.**  
Es el hallazgo más interesante en sentido opuesto al esperado. Con $k=20$ se captura **≈ 21,2 %** de la energía total, y para alcanzar el 80 % se necesitan **≈ 183 componentes** (más de la mitad del rango total disponible, 294). El decaimiento de $\sigma_k$ es brusco solo en los primeros 5–6 valores y luego se aplana. Esto indica que el corpus tiene **rango efectivo alto**: no hay un puñado de temas que dominen, sino un mosaico de subtemas. Notablemente, esto no impide que la SVD funcione para clustering — la *energía capturada* y la *utilidad geométrica para vecindades semánticas* son cosas distintas.

**(4) Solapamiento historia–economía — se cumple.**  
El heatmap `similitud_categorias.png` confirma exactamente esta predicción: el par `historia–economía` tiene la mayor similitud coseno cruzada (**0.210**), por encima de `cultura–historia` (0.192), `ciencia–economía` (0.171) e `historia–ciencia` (0.151). El par más disjunto es `deporte–economía` (0.100), consistente con la intuición. Esto refleja el vocabulario compartido entre artículos históricos y económicos (gobierno, crisis, política, imperio, sociedad).

### Resultados inesperados

- **Componente 1 genérica**: esperábamos que la dirección de mayor varianza correspondiera al tema más representado, pero captura una dimensión de "frecuencia general" del vocabulario. Es un fenómeno conocido en LSA cuando los vectores están L2-normalizados y el corpus es heterogéneo; en la práctica se suele "descartar" la primera componente para clustering, lo que justifica nuestra figura complementaria (C2, C3).
- **Calidad de vecindades a pesar de la baja energía**: pese a capturar solo el 21,2 % de la varianza con $k=20$, las vecindades en el espacio reducido son excelentes. `Real Madrid` tiene como vecinos a `FC Barcelona` (0.98), `Premier League` (0.96), `UEFA` (0.91); `Física` tiene como vecinos a `Electromagnetismo` (0.92), `Mecánica cuántica` (0.91), `Termodinámica` (0.90). Para tareas de recuperación e indexación semántica, la representación es muy útil.
- **Términos cercanos coherentes incluso para palabras polisémicas**: la consulta `inflación` recupera `monetaria, tasa, recesión, crédito` (sentido económico); `gravedad` recupera `agujeros, negros, curvatura, gravitatoria` (sentido físico). LSA resuelve la polisemia *por contexto* del corpus.

### Limitaciones del método

- **Bolsa de palabras**: LSA ignora orden y sintaxis. *No gana* y *gana* tienen vectores casi idénticos.
- **Sensibilidad al preprocesamiento**: la lista de stopwords y los parámetros `min_df`/`max_df`/`max_features` cambian sustancialmente las primeras componentes. Tuvimos que añadir stopwords "enciclopédicas" (`denominado, conocido, llamado, …`) para evitar que dominaran las componentes principales.
- **Escalabilidad**: para corpus mucho más grandes (decenas de miles de documentos × decenas de miles de términos), la SVD densa de NumPy es inviable; en su lugar se usa SVD truncada (`scipy.sparse.linalg.svds`, `sklearn.decomposition.TruncatedSVD`), que sólo calcula los $k$ primeros vectores singulares.
- **Ambigüedad léxica**: una palabra con varios sentidos (e.g., *banco* = entidad financiera o asiento) queda en un único punto del espacio.
- **Linealidad**: LSA sólo captura estructura lineal en la matriz TF-IDF; métodos no lineales (UMAP, t-SNE, embeddings neuronales como `word2vec` o `sentence-transformers`) pueden recuperar estructura más fina.
- **Sesgo del corpus**: los resúmenes de Wikipedia son enciclopédicos y muy formales, lo que facilita artificialmente la separación temática. Un corpus de redes sociales, noticias mezcladas o reviews daría resultados más ruidosos.

### Conclusión

Aplicamos LSA con SVD a un corpus de **294 artículos** de Wikipedia organizados en 5 temas. La descomposición $A = U \Sigma V^T$ sobre la matriz término-documento de TF-IDF ($3\,000 \times 294$) reveló los temas latentes sin supervisión: aunque la primera componente captura un eje genérico, las componentes 2–5 codifican direcciones temáticas interpretables, y el heatmap de similitud confirma cuantitativamente la estructura por categoría (cohesión interna 3× mayor que la similitud cruzada promedio). Tres conclusiones importantes:

1. **La SVD funciona como descubrimiento exploratorio de estructura**, no como compresor de varianza: con $k=20$ (apenas 21 % de la energía) se obtienen vecindades semánticas excelentes.
2. **La hipótesis 3 (energía concentrada) no se cumplió**: el corpus tiene rango efectivo alto, lo que es coherente con su heterogeneidad. Esto es un hallazgo a destacar, no un fracaso del método.
3. **Se confirmó la hipótesis 4 de solapamiento historia–economía** (similitud cruzada 0.210, la más alta entre pares distintos), mostrando que la SVD detecta automáticamente relaciones temáticas no triviales.

El proyecto ilustra cómo el álgebra lineal (descomposición espectral de una matriz dispersa) sirve como herramienta práctica de análisis exploratorio de texto, siempre que se complemente con lectura del corpus y se entienda qué capturan (y qué *no* capturan) las primeras componentes.
